In [1]:
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '3'

In [2]:
%load_ext autoreload
%autoreload 2

import numpy as np
import torch
from torch.utils.data import TensorDataset
from tqdm.auto import tqdm

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

from denoising_diffusion_pytorch.transport import create_transport, Sampler, FlowMatching
from denoising_diffusion_pytorch.continuous_classifier_free_guidance_1d import GaussianDiffusion1D, Trainer1D, Unet1D


In [3]:
from denoising_diffusion_pytorch.dit import DiT_models, DiTCoordPE


Flash attention 4 not found, using torch scaled_dot_product


/home/d.ramos/denoising-diffusion-pytorch-fluid-mechanics/denoising_diffusion_pytorch/fastlinear/modules/nn/norm.py:186: UserWarning: Cannot import apex RMSNorm, switch to vanilla implementation
  warnings.warn("Cannot import apex RMSNorm, switch to vanilla implementation")


In [4]:
seed = 37
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True

In [5]:
def get_split_indices(split_name, split_data, all_conds, atol=1e-4, rtol=1e-12):
    data = split_data[split_name]
    indices = []

    for row in data:
        # Compare this row against all rows in all_conds
        matches = np.all(np.isclose(all_conds, row, atol=atol, rtol=rtol), axis=1)
        found = np.where(matches)[0]

        if len(found) == 0:
            raise KeyError(f"No match found within tolerance for {row}")
        if len(found) > 1:
            raise ValueError(f"Multiple matches found within tolerance for {row}")

        indices.append(found[0])

    return indices

data = np.load("data/aeronef/db_random.npy", allow_pickle=True).item()
field_name_to_predict = "Cp" # "Pressure"
# cp = data["Cp"]
# train_size = 0.8
# training_indices = np.random.choice(cp.shape[0], int(cp.shape[0] * train_size), replace=False)
# test_indices = np.setdiff1d(np.arange(cp.shape[0]), training_indices)
split_data = np.load("data/aeronef/best_train-val-test_split.npy", allow_pickle=True).item()
alpha, vel_inf = data["Alpha"], data["Vinf"]
all_conds = np.stack((alpha, vel_inf), axis=1)
training_indices = get_split_indices("Train", split_data, all_conds=all_conds)
val_indices = get_split_indices("Validation", split_data, all_conds=all_conds)
test_indices = get_split_indices("Test", split_data, all_conds=all_conds)

def load_dataset(indices, norm_coefficients, data):
    aoa = data["Alpha"][indices]
    vinf = data["Vinf"][indices]
    cp = data[field_name_to_predict][indices]
    if "cp_min" not in norm_coefficients:
        norm_coefficients["cp_min"] = cp.min()
        norm_coefficients["cp_max"] = cp.max()
    cp = (cp - norm_coefficients["cp_min"]) / (norm_coefficients["cp_max"] - norm_coefficients["cp_min"])

    if "vinf_mean" not in norm_coefficients:
        norm_coefficients["vinf_mean"] = vinf.mean()
        norm_coefficients["vinf_std"] = vinf.std()
    vinf = (vinf - norm_coefficients["vinf_mean"]) / norm_coefficients["vinf_std"]

    if "aoa_mean" not in norm_coefficients:
        norm_coefficients["aoa_mean"] = aoa.mean()
        norm_coefficients["aoa_std"] = aoa.std()
    aoa = (aoa - norm_coefficients["aoa_mean"]) / norm_coefficients["aoa_std"]

    # pad/truncate to length 27500
    target_len = 27500
    if field_name_to_predict == "Pressure" and cp.shape[1] < target_len:
        pad_width = target_len - cp.shape[1]
        cp = np.pad(cp, ((0, 0), (0, pad_width)), mode='constant', constant_values=0)
    # pad_width = 696 - cp.shape[1]
    # cp = np.pad(cp, ((0, 0), (0, pad_width)), mode='constant', constant_values=0)
    cp_t = torch.from_numpy(cp).float().unsqueeze(1)  # add channel dimension
    # cp_t = cp_t[..., :-1]
    aoa_t = torch.from_numpy(aoa).float()
    mach_t = torch.from_numpy(vinf).float()
    conditions = torch.stack([aoa_t, mach_t], dim=1)
    print(cp.min(), cp.max())
    print(aoa.mean(), aoa.std())
    print(vinf.mean(), vinf.std())
    print(cp_t.shape, conditions.shape)

    return TensorDataset(cp_t, conditions)

coefficients = {}
dataset = load_dataset(training_indices, coefficients, data)
val_dataset = load_dataset(val_indices, coefficients, data)
test_dataset = load_dataset(test_indices, coefficients, data)

0.0 1.0
-2.9269516103754126e-16 1.0
-3.1792750250629484e-16 1.0
torch.Size([704, 1, 691]) torch.Size([704, 2])
0.03302204411669626 0.9997627695857313
-0.019665332622866426 0.9634137687293968
0.00021769792178616336 1.0137332916699355
torch.Size([151, 1, 691]) torch.Size([151, 2])
-0.004481321056569833 0.9997085660619324
-0.010524482971941413 0.9730003939913162
-0.0006613766563713316 0.9745783970358266
torch.Size([152, 1, 691]) torch.Size([152, 2])


In [6]:
# model = Unet1D(
#     dim=128,
#     dim_mults=(1, 2, 4),  # , 8),
#     # flash_attn = False,
#     channels=1,  
#     cond_dim=2,
#     cond_drop_prob=0.5,
#     attn_dim_head=64,
#     attn_heads=8,
#     learn_sigma=False,
#     # self_condition=True,
#     full_attn=True
# )
# model = Unet1D(
#     dim=128,
#     dim_mults=(1, 2, 4),  # , 8),
#     # flash_attn = False,
#     channels=1,  
#     cond_dim=2,
#     cond_drop_prob=0.2,
#     attn_dim_head=64,
#     attn_heads=8,
#     learn_sigma=False,
#     # self_condition=True,
#     full_attn=True,
#     # qknorm=True
# )
# model = DiT_models['DiT-XXS/1'](
#     input_size=dataset.tensors[0].shape[2],
#     cond_dim=2,
#     class_dropout_prob=0.2,
#     in_channels=1,
#     learn_sigma=False,
#     # use_bias=False,
#     use_swiglu=True,
#     use_rope=False,
#     # qk_norm=True,
#     attn_type="vanilla", # window, linear, vanilla
#     # window_size=107,
#     # num_experts=8,
#     # num_experts_per_tok=2
#     mlp_ratio=2.5
# )
coord = torch.tensor(data["Airfoil"])

model = DiTCoordPE(
    coords=coord,
    depth=6,
    hidden_size=128,
    patch_size=1,
    num_heads=4,
    input_size=dataset.tensors[0].shape[2],
    cond_dim=2,
    class_dropout_prob=0.2,
    in_channels=1,
    learn_sigma=False,
    # use_bias=False,
    use_swiglu=True,
    use_rope=True,
    # qk_norm=True,
    attn_type="vanilla",  # window, linear, vanilla
    # window_size=107,
    # num_experts=8,
    # num_experts_per_tok=2
    mlp_ratio=2.5,
)
sampler = Sampler(transport=create_transport(
    # use_cosine_loss=True,
    # use_lognorm=True
))

fm_diffusion = FlowMatching(
    sampler,
    model,
    input_size=dataset.tensors[0].shape[2],
    cond_scale=6,
    num_sampling_steps=500,
    sampling_method="rk4",
    # sampler_timestep_shift=0.2
)

Creating 1D DiT
======== RoPE 1D initialized with shape torch.Size([691, 32]) ========
Creating DiT with 691 patches.


In [7]:
gausian_diffusion = GaussianDiffusion1D(
    model,
    seq_length=dataset.tensors[0].shape[2],
    objective="pred_noise",  # 'pred_noise' or 'pred_x0'
    beta_schedule="cosine",
    sampling_timesteps=1000,
    timesteps=1000,  # number of steps
    # use_cfg_plus_plus=True,
    min_snr_loss_weight=True,
    min_snr_gamma=5
)

In [8]:
from cetaceo.evaluators import RegressionEvaluator


def evaluate_model(samples, test_data):
    cp_true = test_data * (coefficients["cp_max"] - coefficients["cp_min"]) + coefficients["cp_min"]
    evaluator = RegressionEvaluator()
    metrics = evaluator(samples[-1], cp_true)
    evaluator.print_metrics()
    return metrics

def get_last_checkpoint_idx(results_folder):
    import os
    import re

    checkpoint_files = [f for f in os.listdir(results_folder) if re.match(r'model-\d+\.pt', f)]
    if not checkpoint_files:
        return None

    checkpoint_indices = [int(re.findall(r'\d+', f)[0]) for f in checkpoint_files]
    last_checkpoint_idx = max(checkpoint_indices)
    return last_checkpoint_idx

# TODO: sacar el trainer de aqui
def load_and_sample(results_folder, diffusion, test_dataset, cond_scale=6, sample_batch_size=128, compile_model=True, sampling_kwargs={}):
    torch.manual_seed(42)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(42)
    np.random.seed(42)
    torch.set_float32_matmul_precision('high')
    torch.backends.cuda.matmul.allow_tf32 = True
    # results_folder = 'results/aeronef_cp_new_split/FM_unet_L_first'
    trainer = Trainer1D(
        diffusion,
        # 'path/to/your/images',
        dataset=dataset,
        dataset_test=val_dataset,
        train_batch_size=64,
        train_lr=1e-4,
        num_samples=9,
        train_num_steps=10000+4,  # total training steps
        gradient_accumulate_every=1,  # gradient accumulation steps
        ema_decay=0.995,  # exponential moving average decay
        # amp = True,                       # turn on mixed precision
        results_folder=results_folder,  # folder to save results to
        save_and_sample_every=15000,
        # use_lr_scheduler=False,
        max_grad_norm=1.0,
        # use_cpu=True,
        # use_muon=True,
        compile_model=compile_model
    )
    idx = get_last_checkpoint_idx(results_folder)
    print("Loading checkpoint:", idx)
    print("Number of parameters: ", sum(p.numel() for p in trainer.ema.ema_model.parameters()))
    trainer.load(idx)
    trainer.ema.ema_model.eval()
    torch.manual_seed(42)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(42)
    np.random.seed(42)

    test_data, test_parameters = test_dataset.tensors
    test_parameters = test_parameters.cuda()
    batches = torch.split(test_parameters, sample_batch_size)
    predictions = []
    with torch.inference_mode():
        for batch in tqdm(batches):
            # batch = batch.to(model_device)
            batch_pred = trainer.ema.ema_model.sample(batch, return_all_steps=True, cond_scale=cond_scale, **sampling_kwargs).cpu()
            # batch_pred = trainer.ema.ema_model.sample_flow_dpm(batch).cpu()
            print(batch_pred.shape)
            predictions.append(batch_pred)

    samples = torch.cat(predictions, dim=1)
    # samples = torch.cat(predictions, dim=0) # this is for dpm solver
    # samples_rescaled = samples * (coefficients["cp_max"] - coefficients["cp_min"]) + coefficients["cp_min"]
    # evaluate_model(samples_rescaled, test_data)
    return samples

In [9]:
fm_model_dir = 'results/aeronef_cp_good_split/dit_xxs_coord_pe'  #dit_2_xxs_mlp_2.5_v2'

In [11]:
fm_samples = load_and_sample(fm_model_dir, fm_diffusion, test_dataset, cond_scale=2, sample_batch_size=128, sampling_kwargs={"cfg_interval_start": 0.2})
fm_samples = fm_samples.squeeze()
fm_samples.shape

Loading checkpoint: 10
Number of parameters:  1708031
loading from version 3.0.0
Setting loaded learning rate to 1e-06


  0%|          | 0/2 [00:00<?, ?it/s]

torch.Size([500, 128, 1, 691])
torch.Size([500, 24, 1, 691])


torch.Size([500, 152, 691])

In [12]:
# fm_samples = torch.load(f"{fm_model_dir}/test_predictions_ema_rk4.pt")
# samples_rescaled = fm_samples * (coefficients["cp_max"] - coefficients["cp_min"]) + coefficients["cp_min"]
cp = dataset.tensors[0] * (coefficients["cp_max"] - coefficients["cp_min"]) + coefficients["cp_min"]
samples_rescaled = fm_samples * cp.std() + cp.mean()
cp_true = test_dataset.tensors[0].squeeze() * (coefficients["cp_max"] - coefficients["cp_min"]) + coefficients["cp_min"]
evaluator = RegressionEvaluator()
metrics = evaluator(samples_rescaled[-1, :691], cp_true)
# metrics = evaluator(samples_rescaled, cp_true)
evaluator.print_metrics()


Regression evaluator metrics:
mse: 1.0807e-04
rmse: 0.0104
mae: 0.0025
mre: 3.1269%
ae_95: 0.0076
ae_99: 0.0346
r2: 0.9998
l2_error: 0.0143


In [12]:
torch.save(fm_samples, f"{fm_model_dir}/test_predictions_ema_evolutions.pt")

In [13]:
torch.save(fm_samples[-1], f"{fm_model_dir}/test_predictions_ema_cfg2.pt")

In [ ]:
diff_samples = load_and_sample('results/aeronef_cp_new_split/unet_L_bs_128_lr2e-4_repeat', gausian_diffusion, test_dataset, cond_scale=6, sample_batch_size=128, compile_model=True)
diff_samples = diff_samples.squeeze()
diff_samples.shape

Compiling model...
Model compiled
Loading checkpoint: 10
loading from version 2.2.5


  0%|          | 0/2 [00:00<?, ?it/s]

sampling loop time step:   0%|          | 0/1000 [00:00<?, ?it/s]

torch.Size([1001, 128, 1, 691])


sampling loop time step:   0%|          | 0/1000 [00:00<?, ?it/s]

torch.Size([1001, 24, 1, 691])

Regression evaluator metrics:
mse: 1.6200e-04
rmse: 0.0127
mae: 0.0082
mre: 9.8771%
ae_95: 0.0246
ae_99: 0.0477
r2: 0.9996
l2_error: 0.0196


torch.Size([1001, 152, 691])

In [10]:
samples_rescaled = diff_samples * (coefficients["cp_max"] - coefficients["cp_min"]) + coefficients["cp_min"]
cp_true = test_dataset.tensors[0].squeeze() * (coefficients["cp_max"] - coefficients["cp_min"]) + coefficients["cp_min"]
evaluator = RegressionEvaluator()
metrics = evaluator(samples_rescaled[-1], cp_true)
evaluator.print_metrics()


Regression evaluator metrics:
mse: 1.6200e-04
rmse: 0.0127
mae: 0.0082
mre: 9.8771%
ae_95: 0.0246
ae_99: 0.0477
r2: 0.9996
l2_error: 0.0196


In [12]:
torch.save(diff_samples, f"results/aeronef_cp_new_split/unet_L_bs_128_lr2e-4_repeat/test_predictions_evolutions.pt")

# Visualizaciones

In [5]:
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib
matplotlib.rcParams['animation.embed_limit'] = 2**128

def create_animation(airfoil_idx, samples, test_parameters, data, cp_true, save_path=None):
    conditions = test_parameters[airfoil_idx].cpu().numpy()
    airfoil_data = samples[:, airfoil_idx, ...].squeeze().numpy()  
    x_coords = data["Airfoil"][:, 0]

    # Quick version
    fig, ax = plt.subplots()
    ax.set_title(f"Airfoil {airfoil_idx}: α={conditions[0]:.2f}°, VelInf={conditions[1]:.2f}")
    # ax.legend(fontsize=8)
    ax.grid(True)#, alpha=0.3)
    ax.set_xlabel('x')
    ax.set_ylabel('cp')
    # scatter = ax.scatter(x_coords, airfoil_data[0], c='orange', s=1)
    
    scatter_true = ax.scatter([], [], label="true cp", s=1)  # Empty initially
    scatter_pred = ax.scatter(x_coords, airfoil_data[0], label="predicted cp", s=1)

    def animate(i):
        # Update predicted scatter plot data
        
        # On the last frame, add true values
        if i == len(airfoil_data) - 1:
            scatter_true.set_offsets(np.c_[x_coords, cp_true])
            # Update axis limits to show both datasets
            all_y_values = np.concatenate([cp_true, airfoil_data[i]])
            ax.set_ylim(all_y_values.min(), all_y_values.max())
        else:
            scatter_true.set_offsets(np.empty((0, 2)))  # Keep empty for other frames
            # Update axis limits based on current frame only
            ax.set_ylim(airfoil_data[i].min(), airfoil_data[i].max())
        
        scatter_pred.set_offsets(np.c_[x_coords, airfoil_data[i]])
        ax.set_xlim(x_coords.min(), x_coords.max())
        
        return [scatter_true, scatter_pred]

    ani = animation.FuncAnimation(fig, animate, frames=len(airfoil_data), interval=100, repeat=True)

    if save_path is not None:
        with open(save_path, 'w') as f:
            f.write(ani.to_jshtml())

    plt.close()
    return HTML(ani.to_jshtml())

In [6]:
test_data, test_parameters = test_dataset.tensors
cp_true = test_data * (coefficients["cp_max"] - coefficients["cp_min"]) + coefficients["cp_min"]

rescaled_conditions = test_parameters * torch.tensor([
    coefficients["aoa_std"],
    coefficients["vinf_std"]
], device=test_parameters.device) + torch.tensor([
    coefficients["aoa_mean"],
    coefficients["vinf_mean"]
], device=test_parameters.device)

In [ ]:
airfoil_idx = 0
create_animation(
    airfoil_idx=airfoil_idx,
    samples=fm_samples,
    test_parameters=rescaled_conditions,
    data=data,
    cp_true=cp_true[airfoil_idx].squeeze().cpu().numpy(),
    save_path="cp_evolution_fm.html",
)

In [25]:
airfoil_idx = 0
# diff_samples = diff_samples * (coefficients["cp_max"] - coefficients["cp_min"]) + coefficients["cp_min"]
create_animation(
    airfoil_idx=airfoil_idx,
    samples=diff_samples * (coefficients["cp_max"] - coefficients["cp_min"]) + coefficients["cp_min"],
    test_parameters=rescaled_conditions,
    data=data,
    cp_true=cp_true[airfoil_idx].squeeze().cpu().numpy(),
    save_path="cp_evolution_diff_ddim.html",
)

In [ ]:
torch.save(fm_samples, f"{fm_model_dir}/test_predictions_evolutions.pt")

In [7]:
# model_dir = 'results/aeronef_cp_new_split/FM_unet_L_euler_500_300k'
model_dir = 'results/aeronef_cp_good_split/dit_xxs_mlp_ratio_2_5'
# model_dir = 'results/aeronef_cp_new_split/unet_L_bs_128_lr2e-4_repeat'
samples = torch.load(f"{model_dir}/test_predictions_ema_evolutions.pt", weights_only=True, map_location="cpu")
# samples = samples * (coefficients["cp_max"] - coefficients["cp_min"]) + coefficients["cp_min"]
# cp = dataset.tensors[0] * (coefficients["cp_max"] - coefficients["cp_min"]) + coefficients["cp_min"]
samples = samples * cp_true.std() + cp_true.mean()
samples = samples[:, :, :691]
airfoil_idx = 0
test_parameters = rescaled_conditions
samples.shape

torch.Size([500, 152, 691])

In [9]:
samples[-1, 0].min(), cp_true[0].min()

(tensor(-1.6078), tensor(-1.5895))

In [18]:
test_parameters[40:60]

tensor([[8.7800e+00, 5.0797e+01],
        [1.2171e-07, 1.7953e+02],
        [8.0000e+00, 5.0663e+01],
        [3.9600e+00, 2.8447e+02],
        [8.0000e+00, 8.2407e+01],
        [3.4500e+00, 2.4340e+02],
        [3.4500e+00, 2.6262e+02],
        [5.5800e+00, 1.4074e+02],
        [4.1900e+00, 1.4663e+02],
        [7.2000e+00, 2.1213e+02],
        [4.1700e+00, 1.7770e+02],
        [7.2000e+00, 1.5315e+02],
        [3.4500e+00, 2.5954e+02],
        [4.1900e+00, 1.9244e+02],
        [4.1900e+00, 2.6262e+02],
        [7.2000e+00, 2.4613e+02],
        [3.4500e+00, 7.9433e+01],
        [8.0000e+00, 1.7177e+02],
        [2.0400e+00, 1.1156e+01],
        [6.8500e+00, 1.7177e+02]], dtype=torch.float64)

In [19]:
from manim import *
import numpy as np

config.media_width = "75%"
config.verbosity = "WARNING"

class AirfoilAnimation(Scene):
    def construct(self):
        # Get data
        airfoil_idx = 50  # Change this to select different airfoil
        conditions = test_parameters[airfoil_idx].cpu().numpy()
        airfoil_data = samples[:, airfoil_idx, ...].squeeze().numpy()
        print(airfoil_data.min(), airfoil_data.max())
        final_cp_true = cp_true[airfoil_idx].squeeze().cpu().numpy()
        x_coords = data["Airfoil"][:, 0]#.numpy()
        
        # Create title
        title = Text(
            # f"Airfoil {airfoil_idx}: α={conditions[0]:.2f}°, VelInf={conditions[1]:.2f}",
            f"α={conditions[0]:.2f}°, VelInf={conditions[1]:.2f}",
            font_size=24
        ).to_edge(UP)
        self.add(title)
        
        # Set up initial axes
        y_min_init = airfoil_data[0].min()
        y_max_init = airfoil_data[0].max()
        y_range_init = y_max_init - y_min_init
        y_padding_init = y_range_init * 0.1
        
        axes = Axes(
            # x_range=[x_coords.min(), x_coords.max(), (x_coords.max() - x_coords.min()) / 5],
            x_range=[0, 1],
            y_range=[y_min_init - y_padding_init, y_max_init + y_padding_init, y_range_init / 4],
            x_length=10,
            y_length=5,
            axis_config={"include_tip": False, "include_numbers": True},
            x_axis_config={"decimal_number_config": {"num_decimal_places": 2}},
            y_axis_config={"decimal_number_config": {"num_decimal_places": 2}},
        ).shift(DOWN * 0.5)
        
        x_label = axes.get_x_axis_label("x", edge=RIGHT, direction=RIGHT, buff=0.3)
        y_label = axes.get_y_axis_label("cp", edge=LEFT, direction=LEFT, buff=0.3)
        
        self.add(axes, x_label, y_label)
        
        # Create dots once
        dots = VGroup(*[Dot(radius=0.02, color=ORANGE) for _ in x_coords])
        
        # Add legend
        legend_pred = VGroup(
            Dot(radius=0.05, color=ORANGE),
            Text("predicted cp", font_size=20)
        ).arrange(RIGHT, buff=0.1).to_corner(UR, buff=0.5)
        
        self.add(dots)
        
        # Initial position
        for dot, x, y in zip(dots, x_coords, airfoil_data[0]):
            dot.move_to(axes.c2p(x, y))
        
        # Animate through frames - sample every few frames for speed
        n_frames = 50
        frame_step = max(1, len(airfoil_data) // n_frames)  # Show ~n_frames frames total
        # show the final frames, which are the important ones
        frame_range = list(range(frame_step, len(airfoil_data), frame_step)) + list(range(frame_step * (len(airfoil_data) // frame_step), len(airfoil_data))) + [-1]
        for i in frame_range:
            # Calculate new y range for this frame
            y_min_new = airfoil_data[i].min()
            y_max_new = airfoil_data[i].max()
            y_range_new = y_max_new - y_min_new
            y_padding_new = y_range_new * 0.1
            
            # Create new axes with updated range
            new_axes = Axes(
                # x_range=[x_coords.min(), x_coords.max(), (x_coords.max() - x_coords.min()) / 5],
                x_range=[0, 1],
                y_range=[y_min_new - y_padding_new, y_max_new + y_padding_new, y_range_new / 4],
                x_length=10,
                y_length=5,
                axis_config={"include_tip": False, "include_numbers": True},
                x_axis_config={"decimal_number_config": {"num_decimal_places": 2}},
                y_axis_config={"decimal_number_config": {"num_decimal_places": 2}},
            ).shift(DOWN * 0.5)
            
            new_x_label = new_axes.get_x_axis_label("x", edge=RIGHT, direction=RIGHT, buff=0.3)
            new_y_label = new_axes.get_y_axis_label("cp", edge=LEFT, direction=LEFT, buff=0.3)
            
            # Update dot positions for new axes
            new_dots = VGroup(*[Dot(radius=0.02, color=ORANGE) for _ in x_coords])
            for dot, x, y in zip(new_dots, x_coords, airfoil_data[i]):
                dot.move_to(new_axes.c2p(x, y))
            
            # Animate transformation
            self.play(
                Transform(axes, new_axes),
                Transform(x_label, new_x_label),
                Transform(y_label, new_y_label),
                Transform(dots, new_dots),
                run_time=0.2,
                rate_func=linear
            )
        
        # After animation, show true values
        self.wait(0.3)
        
        # Update axes to include true values
        all_y_values = np.concatenate([airfoil_data[-1], final_cp_true])
        y_min_final = all_y_values.min()
        y_max_final = all_y_values.max()
        y_range_final = y_max_final - y_min_final
        y_padding_final = y_range_final * 0.1
        
        final_axes = Axes(
            x_range=[x_coords.min(), x_coords.max(), (x_coords.max() - x_coords.min()) / 5],
            y_range=[y_min_final - y_padding_final, y_max_final + y_padding_final, y_range_final / 4],
            x_length=10,
            y_length=5,
            axis_config={"include_tip": False, "include_numbers": True},
            x_axis_config={"decimal_number_config": {"num_decimal_places": 2}},
            y_axis_config={"decimal_number_config": {"num_decimal_places": 2}},
        ).shift(DOWN * 0.5)
        
        # Add true cp scatter plot
        true_dots = VGroup(*[
            Dot(final_axes.c2p(float(x), float(y)), radius=0.02, color=BLUE) 
            for x, y in zip(x_coords, final_cp_true)
        ])
        
        # Update legend
        legend_true = VGroup(
            Dot(radius=0.05, color=BLUE),
            Text("true cp", font_size=20)
        ).arrange(RIGHT, buff=0.1).next_to(legend_pred, DOWN, aligned_edge=LEFT, buff=0.2)
        
        self.play(
            Transform(axes, final_axes),
            FadeIn(true_dots),
            FadeIn(legend_pred, legend_true),
            run_time=0.7
        )
        self.wait(1)

# Render the animation
scene = AirfoilAnimation()
scene.render()

-3.2607253 1.9533828
